# Vivaldi Antenna — Tapered Slot Radiation & Broadband Performance

A **Vivaldi antenna** (exponentially tapered slot antenna) is a broadband, endfire travelling-wave antenna formed by two flared metallic conductors separated by an exponentially widening slot.

## Key ideas

| Concept | Description |
|---------|-------------|
| Exponential taper | Slot width follows $w(x) = c_1 e^{Rx} + c_2$ |
| Endfire radiation | Main beam along the taper axis ($+x$) |
| Broadband | Typical 3:1 to 10:1 bandwidth |
| Travelling-wave | Energy propagates along slot, radiating where slot width $\approx \lambda/2$ |
| Taper rate $R$ | Controls beamwidth, gain, and impedance bandwidth |

## Exponential Taper Equation

$$w(x) = c_1 e^{Rx} + c_2$$

where $c_1 = (W_a - W_t)/(e^{RL} - 1)$ and $c_2 = (W_t e^{RL} - W_a)/(e^{RL} - 1)$,
with throat width $W_t$, aperture width $W_a$, and slot length $L$.

This notebook visualises the **geometry**, **field distributions**, **radiation patterns**, and **broadband characteristics** of the Vivaldi antenna.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import (
    FloatSlider, IntSlider, Dropdown, Checkbox,
    HBox, VBox, Output, Label
)
import ipywidgets as widgets
from IPython.display import display, clear_output

c0 = 3e8

## 1 — Antenna Geometry & Taper Profile

The Vivaldi taper follows an exponential curve:
$$w(x) = c_1\,e^{Rx} + c_2$$

- **Throat width** $W_t$: narrow end near the feed (typically $\ll \lambda$)
- **Aperture width** $W_a$: wide end where radiation occurs ($\approx \lambda/2$ at lowest frequency)
- **Taper rate** $R$: exponential growth rate — higher $R$ = more abrupt flare
- **Slot length** $L$: overall antenna length

Adjust the sliders to see how the taper profile and physical outline change.

In [2]:
w_gL  = FloatSlider(value=4.0, min=1.0, max=8.0, step=0.1,
                    description='Length L/λ', continuous_update=False,
                    style={'description_width': '100px'})
w_gWa = FloatSlider(value=1.5, min=0.3, max=3.0, step=0.1,
                    description='Aperture W_a/λ', continuous_update=False,
                    style={'description_width': '100px'})
w_gR  = FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05,
                    description='Taper rate R', continuous_update=False,
                    style={'description_width': '100px'})
w_gWt = FloatSlider(value=0.05, min=0.01, max=0.3, step=0.01,
                    description='Throat W_t/λ', continuous_update=False,
                    style={'description_width': '100px'})
out_geom = Output()

def vivaldi_taper(x, L, Wa, Wt, R):
    """Return half-width of slot at position x (0..L)."""
    eRL = np.exp(R * L)
    c1 = (Wa - Wt) / (eRL - 1 + 1e-30)
    c2 = (Wt * eRL - Wa) / (eRL - 1 + 1e-30)
    return (c1 * np.exp(R * x) + c2) / 2  # half-width

def draw_geometry(change=None):
    L = w_gL.value; Wa = w_gWa.value
    Wt = w_gWt.value; R = w_gR.value
    x = np.linspace(0, L, 500)
    hw = vivaldi_taper(x, L, Wa, Wt, R)

    with out_geom:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # antenna outline
        substrate_h = max(Wa * 0.8, hw.max() * 1.3)
        ax1.fill_between(x, -substrate_h, -hw, color='goldenrod', alpha=0.4, label='Conductor')
        ax1.fill_between(x, hw, substrate_h, color='goldenrod', alpha=0.4)
        ax1.fill_between(x, -hw, hw, color='lightskyblue', alpha=0.3, label='Slot')
        ax1.plot(x, hw, 'k-', lw=2)
        ax1.plot(x, -hw, 'k-', lw=2)

        # annotations
        ax1.annotate('', xy=(0, Wt/2), xytext=(0, -Wt/2),
                     arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
        ax1.text(-0.3, 0, f'W_t={Wt:.2f}λ', fontsize=8, color='red', ha='right')

        ax1.annotate('', xy=(L, hw[-1]), xytext=(L, -hw[-1]),
                     arrowprops=dict(arrowstyle='<->', color='blue', lw=1.5))
        ax1.text(L + 0.15, 0, f'W_a={Wa:.2f}λ', fontsize=8, color='blue')

        ax1.annotate('', xy=(L, -substrate_h - 0.15), xytext=(0, -substrate_h - 0.15),
                     arrowprops=dict(arrowstyle='<->', color='green', lw=1.5))
        ax1.text(L/2, -substrate_h - 0.35, f'L={L:.1f}λ', fontsize=8,
                 color='green', ha='center')

        ax1.scatter([0], [0], color='red', s=80, zorder=5, label='Feed')
        ax1.arrow(L, 0, 0.6, 0, head_width=0.15, head_length=0.15,
                  fc='yellow', ec='k', zorder=5)
        ax1.text(L + 0.8, 0.05, 'Endfire', fontsize=9)

        ax1.set_xlabel('x / λ'); ax1.set_ylabel('y / λ')
        ax1.set_title(f'Vivaldi geometry   R={R:.2f}  L={L:.1f}λ  W_a={Wa:.1f}λ')
        ax1.set_aspect('equal')
        ax1.legend(loc='upper left', fontsize=8)
        ax1.grid(alpha=0.2)

        # taper profile
        for Ri in [0.1, 0.3, 0.5, 1.0, 1.5, 2.0]:
            hw_i = vivaldi_taper(x, L, Wa, Wt, Ri)
            alpha = 1.0 if abs(Ri - R) < 0.01 else 0.3
            lw_i = 2.5 if abs(Ri - R) < 0.01 else 1.0
            ax2.plot(x, 2 * hw_i, lw=lw_i, alpha=alpha, label=f'R={Ri:.1f}')
        ax2.set_xlabel('x / λ'); ax2.set_ylabel('Slot width w(x) / λ')
        ax2.set_title('Taper profiles for different R')
        ax2.legend(fontsize=7); ax2.grid(alpha=0.3)

        plt.tight_layout(); plt.show()

for w in [w_gL, w_gWa, w_gR, w_gWt]:
    w.observe(draw_geometry, names='value')
draw_geometry()
display(VBox([HBox([w_gL, w_gWa]), HBox([w_gR, w_gWt]), out_geom]))

## 2 — Current Distribution on Flares

Surface currents flow along the inner edges of both conductors.  
Current is strongest at the **throat** (near the feed) and decays as the slot widens — energy progressively couples into radiation.

The travelling-wave current can be modelled as:
$$J_s(x) \propto e^{-\alpha(x)\,x}\,\cos\!\bigl(kx - \omega t\bigr)$$

where $\alpha(x)$ increases with slot width (more radiation leakage).

In [ ]:
w_cL  = FloatSlider(value=4.0, min=1.0, max=8.0, step=0.1,
                    description='Length L/λ', continuous_update=False,
                    style={'description_width': '100px'})
w_cR  = FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05,
                    description='Taper rate R', continuous_update=False,
                    style={'description_width': '100px'})
w_ct  = FloatSlider(value=0.0, min=0.0, max=2.0, step=0.05,
                    description='Time t/T', continuous_update=False,
                    style={'description_width': '100px'})
out_cur = Output()

def draw_current(change=None):
    L = w_cL.value; R = w_cR.value; t = w_ct.value
    Wa = 1.5; Wt = 0.05
    k = 2 * np.pi; omega = 2 * np.pi
    x = np.linspace(0, L, 500)
    hw = vivaldi_taper(x, L, Wa, Wt, R)

    # leakage rate increases with slot width
    alpha = 0.3 * (2 * hw)  # wider slot → more radiation
    cumulative_alpha = np.cumsum(alpha) * (x[1] - x[0])
    envelope = np.exp(-cumulative_alpha)
    J = envelope * np.cos(k * x - omega * t)

    with out_cur:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

        # instantaneous current
        ax1.plot(x, J, lw=2, color='steelblue', label='J_s(x,t)')
        ax1.plot(x, envelope, '--', lw=1.5, color='tomato', label='Envelope')
        ax1.plot(x, -envelope, '--', lw=1.5, color='tomato')
        ax1.fill_between(x, J, alpha=0.15, color='steelblue')
        ax1.set_ylabel('Surface current J_s')
        ax1.set_title(f'Current on flare edges   R={R:.2f}  L={L:.1f}λ  t={t:.2f}T')
        ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
        ax1.set_ylim(-1.2, 1.2)

        # current mapped onto antenna shape
        colours = plt.cm.RdBu_r((J + 1) / 2)
        for i in range(len(x) - 1):
            ax2.fill_between(x[i:i+2], hw[i:i+2], hw[i:i+2] + 0.04,
                             color=colours[i])
            ax2.fill_between(x[i:i+2], -hw[i:i+2] - 0.04, -hw[i:i+2],
                             color=colours[i])
        ax2.plot(x, hw, 'k-', lw=1.5)
        ax2.plot(x, -hw, 'k-', lw=1.5)
        ax2.fill_between(x, -hw, hw, color='lightskyblue', alpha=0.15)
        ax2.set_xlabel('x / λ'); ax2.set_ylabel('y / λ')
        ax2.set_title('Current distribution mapped on antenna')
        ax2.set_aspect('equal')

        plt.tight_layout(); plt.show()

for w in [w_cL, w_cR, w_ct]:
    w.observe(draw_current, names='value')
draw_current()
display(VBox([HBox([w_cL, w_cR, w_ct]), out_cur]))

## 3 — E-field in the Slot

The transverse electric field $E_y$ across the slot transitions from a **bound slot-line mode** at the throat to a **radiated wave** at the aperture.

As the slot widens, the field spreads and its phase velocity approaches $c$ — the wave "peels off" into free space.

$$E_y(x,y,t) \propto \frac{\exp(-|y|/w(x))}{\sqrt{w(x)}}\,\cos(kx - \omega t)\,e^{-\alpha_{cum}(x)}$$

In [ ]:
w_fL  = FloatSlider(value=4.0, min=1.0, max=8.0, step=0.1,
                    description='Length L/λ', continuous_update=False,
                    style={'description_width': '100px'})
w_fR  = FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05,
                    description='Taper rate R', continuous_update=False,
                    style={'description_width': '100px'})
w_ft  = FloatSlider(value=0.0, min=0.0, max=2.0, step=0.05,
                    description='Time t/T', continuous_update=False,
                    style={'description_width': '100px'})
out_field = Output()

def draw_efield(change=None):
    L = w_fL.value; R = w_fR.value; t = w_ft.value
    Wa = 1.5; Wt = 0.05
    k = 2 * np.pi; omega = 2 * np.pi

    xg = np.linspace(0, L, 300)
    ymax = Wa * 0.8
    yg = np.linspace(-ymax, ymax, 250)
    X, Y = np.meshgrid(xg, yg)

    hw = vivaldi_taper(X.ravel(), L, Wa, Wt, R).reshape(X.shape)
    hw_1d = vivaldi_taper(xg, L, Wa, Wt, R)

    # leakage
    alpha_1d = 0.3 * (2 * hw_1d)
    cum_alpha_1d = np.cumsum(alpha_1d) * (xg[1] - xg[0])
    cum_alpha = np.interp(X.ravel(), xg, cum_alpha_1d).reshape(X.shape)

    # slot field: confined transversely, decaying along x
    slot_width = 2 * hw + 1e-6
    confinement = np.exp(-np.abs(Y) / (slot_width / 2 + 1e-6)) / np.sqrt(slot_width + 1e-6)
    mask_slot = np.abs(Y) < hw + 0.3  # field extends slightly beyond edges
    Ey = confinement * np.cos(k * X - omega * t) * np.exp(-cum_alpha) * mask_slot

    with out_field:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(13, 5))

        vmax = np.percentile(np.abs(Ey[Ey != 0]), 97) if np.any(Ey != 0) else 1
        ax.pcolormesh(X, Y, Ey, cmap='RdBu_r', shading='auto',
                      vmin=-vmax, vmax=vmax, rasterized=True)
        # antenna edges
        hw_plot = vivaldi_taper(xg, L, Wa, Wt, R)
        ax.plot(xg, hw_plot, 'k-', lw=2)
        ax.plot(xg, -hw_plot, 'k-', lw=2)
        ax.scatter([0], [0], color='red', s=60, zorder=5)

        ax.set_xlabel('x / λ'); ax.set_ylabel('y / λ')
        ax.set_title(f'E_y field in slot   R={R:.2f}  t={t:.2f}T')
        ax.set_aspect('equal')

        plt.tight_layout(); plt.show()

for w in [w_fL, w_fR, w_ft]:
    w.observe(draw_efield, names='value')
draw_efield()
display(VBox([HBox([w_fL, w_fR, w_ft]), out_field]))

## 4 — Radiation Pattern (E-plane & H-plane)

The Vivaldi radiates **endfire** — main beam along $+x$ (the taper axis).

The far-field pattern is computed by integrating the aperture field:
$$E(\theta) = \int_{-W_a/2}^{W_a/2} E_{ap}(y)\,e^{\,jky\sin\theta}\,dy$$

where $E_{ap}(y)$ is the field distribution across the aperture opening.

- **E-plane** (plane of the slot, $xz$-plane): pattern shaped by slot length $L$
- **H-plane** ($xy$-plane): pattern shaped by aperture width $W_a$

In [ ]:
w_pWa = FloatSlider(value=1.5, min=0.3, max=3.0, step=0.1,
                    description='W_a/λ', continuous_update=False,
                    style={'description_width': '100px'})
w_pL  = FloatSlider(value=4.0, min=1.0, max=8.0, step=0.1,
                    description='L/λ', continuous_update=False,
                    style={'description_width': '100px'})
w_pR  = FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05,
                    description='Taper rate R', continuous_update=False,
                    style={'description_width': '100px'})
out_pat = Output()

def vivaldi_pattern(theta, Wa, L, R, plane='E'):
    """Compute normalised Vivaldi pattern in E or H plane."""
    k = 2 * np.pi
    if plane == 'H':
        # H-plane: aperture integration with tapered distribution
        Ny = 200
        y = np.linspace(-Wa/2, Wa/2, Ny)
        # aperture field decays from centre
        Eap = np.exp(-2 * np.abs(y) / (Wa / 2 + 1e-6))
        # array factor by integration
        pat = np.zeros_like(theta, dtype=complex)
        for i, th in enumerate(theta):
            pat[i] = np.trapz(Eap * np.exp(1j * k * y * np.sin(th)), y)
        pat = np.abs(pat)
    else:
        # E-plane: slot-length contribution
        # travelling-wave antenna model: each dx segment radiates
        Nx = 300
        x = np.linspace(0, L, Nx)
        Wt = 0.05
        hw = vivaldi_taper(x, L, Wa, Wt, R)
        alpha = 0.3 * (2 * hw)
        cum_alpha = np.cumsum(alpha) * (x[1] - x[0])
        amp = np.exp(-cum_alpha)
        pat = np.zeros_like(theta, dtype=complex)
        for i, th in enumerate(theta):
            # endfire: theta=0 is along +x
            pat[i] = np.trapz(amp * np.exp(1j * k * x * (1 - np.cos(th))), x)
        pat = np.abs(pat)

    pat /= pat.max() + 1e-30
    return pat

def draw_pattern(change=None):
    Wa = w_pWa.value; L = w_pL.value; R = w_pR.value

    theta = np.linspace(-np.pi, np.pi, 720)
    pat_E = vivaldi_pattern(theta, Wa, L, R, 'E')
    pat_H = vivaldi_pattern(theta, Wa, L, R, 'H')

    theta_deg = np.degrees(theta)
    pat_E_dB = 20 * np.log10(pat_E + 1e-10)
    pat_H_dB = 20 * np.log10(pat_H + 1e-10)

    with out_pat:
        clear_output(wait=True)
        fig = plt.figure(figsize=(14, 5.5))
        ax1 = fig.add_subplot(121, projection='polar')
        ax2 = fig.add_subplot(122)

        ax1.plot(theta, pat_E, lw=2, color='tomato', label='E-plane')
        ax1.plot(theta, pat_H, lw=2, color='steelblue', label='H-plane')
        ax1.set_ylim(0, 1.05)
        ax1.set_title('Polar pattern', pad=15)
        ax1.legend(loc='upper right', fontsize=8)

        ax2.plot(theta_deg, pat_E_dB, lw=2, color='tomato', label='E-plane')
        ax2.plot(theta_deg, pat_H_dB, lw=2, color='steelblue', label='H-plane')
        ax2.set_xlim(-180, 180); ax2.set_ylim(-40, 3)
        ax2.set_xlabel('θ (°)'); ax2.set_ylabel('Pattern (dB)')
        ax2.set_title(f'Radiation pattern   W_a={Wa:.1f}λ  L={L:.1f}λ  R={R:.2f}')
        ax2.axvline(0, color='gray', ls=':', lw=0.8, label='Endfire')
        ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

        plt.tight_layout(); plt.show()

for w in [w_pWa, w_pL, w_pR]:
    w.observe(draw_pattern, names='value')
draw_pattern()
display(VBox([HBox([w_pWa, w_pL, w_pR]), out_pat]))

## 5 — 3D Radiation Pattern

Full 3D visualisation of the Vivaldi radiation pattern.  
The pattern is computed on a $(\theta, \phi)$ grid and mapped to Cartesian coordinates for `plot_surface`.

In [ ]:
w_3Wa = FloatSlider(value=1.5, min=0.3, max=3.0, step=0.1,
                    description='W_a/λ', continuous_update=False,
                    style={'description_width': '100px'})
w_3L  = FloatSlider(value=4.0, min=1.0, max=8.0, step=0.1,
                    description='L/λ', continuous_update=False,
                    style={'description_width': '100px'})
w_3R  = FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05,
                    description='Taper rate R', continuous_update=False,
                    style={'description_width': '100px'})
out_3d = Output()

def draw_3d_pattern(change=None):
    Wa = w_3Wa.value; L = w_3L.value; R = w_3R.value

    theta = np.linspace(0, np.pi, 90)
    phi = np.linspace(0, 2 * np.pi, 180)
    TH, PH = np.meshgrid(theta, phi)

    # E-plane pattern (theta from endfire axis)
    pat_E = vivaldi_pattern(theta, Wa, L, R, 'E')
    pat_H = vivaldi_pattern(theta, Wa, L, R, 'H')

    # combine: E-plane in xz, H-plane in xy
    pat_3d = np.sqrt((pat_E[np.newaxis, :] * np.cos(PH)**2) +
                     (pat_H[np.newaxis, :] * np.sin(PH)**2))
    pat_3d /= pat_3d.max() + 1e-30

    X = pat_3d * np.sin(TH) * np.cos(PH)
    Y = pat_3d * np.sin(TH) * np.sin(PH)
    Z = pat_3d * np.cos(TH)

    with out_3d:
        clear_output(wait=True)
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')

        ax.plot_surface(X, Y, Z, facecolors=plt.cm.hot(pat_3d),
                        alpha=0.85, rstride=1, cstride=1, linewidth=0)

        ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
        ax.set_title(f'3D pattern   W_a={Wa:.1f}λ  L={L:.1f}λ  R={R:.2f}')
        ax.set_box_aspect([1, 1, 1])

        plt.tight_layout(); plt.show()

for w in [w_3Wa, w_3L, w_3R]:
    w.observe(draw_3d_pattern, names='value')
draw_3d_pattern()
display(VBox([HBox([w_3Wa, w_3L, w_3R]), out_3d]))

## 6 — Bandwidth & Impedance

The Vivaldi achieves broadband matching through its **gradual taper** — a smooth impedance transition from the narrow feed line ($Z_{feed} \approx 50\,\Omega$) to free space ($\eta_0 = 377\,\Omega$).

The reflection coefficient at each frequency depends on how well the taper matches:
$$|\Gamma(f)| \approx \frac{|Z_{in}(f) - Z_0|}{|Z_{in}(f) + Z_0|}$$

A longer, gentler taper → broader bandwidth.

In [ ]:
w_iR  = FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05,
                    description='Taper rate R', continuous_update=False,
                    style={'description_width': '100px'})
w_iL  = FloatSlider(value=4.0, min=1.0, max=8.0, step=0.1,
                    description='Length L/λ₀', continuous_update=False,
                    style={'description_width': '100px'})
w_iZ  = FloatSlider(value=50.0, min=30.0, max=100.0, step=5.0,
                    description='Z_feed (Ω)', continuous_update=False,
                    style={'description_width': '100px'})
out_imp = Output()

def draw_impedance(change=None):
    R = w_iR.value; L = w_iL.value; Zfeed = w_iZ.value
    Z0 = 377  # free space

    f_ratio = np.linspace(0.3, 3.0, 500)  # f / f0

    # at each frequency, L in wavelengths changes: L_eff = L * f/f0
    L_eff = L * f_ratio
    # taper quality factor: longer electrical length → better match
    tau = R * L_eff
    # smooth impedance transition model
    transition = 1 - np.exp(-0.5 * tau)
    Rin = Zfeed + (Z0 - Zfeed) * transition
    # reactance: residual from imperfect taper — oscillatory, decaying
    Xin = 40 * np.exp(-0.3 * tau) * np.sin(2 * np.pi * L_eff)
    Zin = Rin + 1j * Xin

    Gamma = np.abs((Zin - Zfeed) / (Zin + Zfeed))
    S11_dB = 20 * np.log10(Gamma + 1e-10)

    with out_imp:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(f_ratio, S11_dB, lw=2, color='steelblue')
        ax1.axhline(-10, color='tomato', ls='--', lw=1, label='−10 dB')
        ax1.axhline(-15, color='gold', ls=':', lw=1, label='−15 dB')
        # bandwidth markers
        bw_mask = S11_dB < -10
        if np.any(bw_mask):
            f_lo = f_ratio[bw_mask][0]
            f_hi = f_ratio[bw_mask][-1]
            ax1.axvspan(f_lo, f_hi, alpha=0.1, color='green')
            ax1.set_title(f'|S11|   BW (−10 dB): {f_lo:.2f}–{f_hi:.2f} f₀  '
                          f'({f_hi/f_lo:.1f}:1)')
        else:
            ax1.set_title('|S11|')
        ax1.set_xlabel('f / f₀'); ax1.set_ylabel('|S11| (dB)')
        ax1.set_ylim(-35, 0)
        ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

        ax2.plot(f_ratio, Rin, lw=2, color='steelblue', label='R_in')
        ax2.plot(f_ratio, Xin, lw=2, color='tomato', label='X_in')
        ax2.axhline(Zfeed, color='gray', ls=':', lw=1, label=f'Z_feed={Zfeed:.0f}Ω')
        ax2.set_xlabel('f / f₀'); ax2.set_ylabel('Impedance (Ω)')
        ax2.set_title(f'Input impedance   R={R:.2f}  L={L:.1f}λ₀')
        ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

        plt.tight_layout(); plt.show()

for w in [w_iR, w_iL, w_iZ]:
    w.observe(draw_impedance, names='value')
draw_impedance()
display(VBox([HBox([w_iR, w_iL, w_iZ]), out_imp]))

## 7 — Gain vs Frequency

The Vivaldi gain increases with frequency because the **electrical size** of the antenna grows:
$$G(f) \approx \frac{4\pi A_{eff}(f)}{\lambda^2} = \frac{4\pi A_{eff}(f) f^2}{c^2}$$

At low frequencies the antenna is electrically small → low gain.  
At high frequencies, the taper is many wavelengths long → high directivity.

Typical Vivaldi: 3–12 dBi over a 3:1 bandwidth.

In [ ]:
w_gfR  = FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05,
                     description='Taper rate R', continuous_update=False,
                     style={'description_width': '100px'})
w_gfL  = FloatSlider(value=4.0, min=1.0, max=8.0, step=0.1,
                     description='Length L/λ₀', continuous_update=False,
                     style={'description_width': '100px'})
w_gfWa = FloatSlider(value=1.5, min=0.3, max=3.0, step=0.1,
                     description='W_a/λ₀', continuous_update=False,
                     style={'description_width': '100px'})
out_gain = Output()

def draw_gain(change=None):
    R = w_gfR.value; L = w_gfL.value; Wa = w_gfWa.value

    f_ratio = np.linspace(0.3, 3.0, 200)
    gains = []
    hpbw_E = []
    hpbw_H = []

    for fr in f_ratio:
        # at frequency fr*f0, wavelength = 1/fr
        # so L_eff = L * fr, Wa_eff = Wa * fr in wavelengths
        L_eff = L * fr
        Wa_eff = Wa * fr
        theta = np.linspace(-np.pi, np.pi, 720)
        pe = vivaldi_pattern(theta, Wa_eff, L_eff, R, 'E')
        ph = vivaldi_pattern(theta, Wa_eff, L_eff, R, 'H')

        # directivity estimate from E-plane beamwidth
        pe_dB = 20 * np.log10(pe + 1e-10)
        ph_dB = 20 * np.log10(ph + 1e-10)
        hp_e = np.where(pe_dB >= -3)[0]
        hp_h = np.where(ph_dB >= -3)[0]
        bw_e = np.degrees(theta[hp_e[-1]] - theta[hp_e[0]]) if len(hp_e) >= 2 else 180
        bw_h = np.degrees(theta[hp_h[-1]] - theta[hp_h[0]]) if len(hp_h) >= 2 else 360
        hpbw_E.append(bw_e)
        hpbw_H.append(bw_h)

        # Kraus approximation: D ≈ 41253 / (HPBW_E * HPBW_H)
        D = 41253 / (abs(bw_e) * abs(bw_h) + 1e-6)
        D = min(D, 100)  # cap
        gains.append(10 * np.log10(D + 1e-10))

    with out_gain:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(f_ratio, gains, lw=2, color='gold')
        ax1.set_xlabel('f / f₀'); ax1.set_ylabel('Gain (dBi)')
        ax1.set_title(f'Gain vs frequency   R={R:.2f}  L={L:.1f}λ₀  W_a={Wa:.1f}λ₀')
        ax1.grid(alpha=0.3)
        ax1.set_ylim(0, 20)

        ax2.plot(f_ratio, hpbw_E, lw=2, color='tomato', label='E-plane HPBW')
        ax2.plot(f_ratio, hpbw_H, lw=2, color='steelblue', label='H-plane HPBW')
        ax2.set_xlabel('f / f₀'); ax2.set_ylabel('HPBW (°)')
        ax2.set_title('Beamwidth vs frequency')
        ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

        plt.tight_layout(); plt.show()

for w in [w_gfR, w_gfL, w_gfWa]:
    w.observe(draw_gain, names='value')
draw_gain()
display(VBox([HBox([w_gfR, w_gfL, w_gfWa]), out_gain]))

## 8 — Taper Rate Effect on Beamwidth

The taper rate $R$ controls the trade-off between:
- **Low $R$** (gentle taper): wider bandwidth, broader beam, lower gain
- **High $R$** (aggressive taper): narrower bandwidth, narrower beam, higher gain

Below: HPBW in both planes and estimated directivity as $R$ varies.

In [ ]:
w_trL  = FloatSlider(value=4.0, min=1.0, max=8.0, step=0.1,
                     description='Length L/λ', continuous_update=False,
                     style={'description_width': '100px'})
w_trWa = FloatSlider(value=1.5, min=0.3, max=3.0, step=0.1,
                     description='W_a/λ', continuous_update=False,
                     style={'description_width': '100px'})
out_taper = Output()

def draw_taper_effect(change=None):
    L = w_trL.value; Wa = w_trWa.value
    Rs = np.linspace(0.05, 2.0, 40)
    hpbw_E = []; hpbw_H = []; gains = []
    theta = np.linspace(-np.pi, np.pi, 720)

    for Ri in Rs:
        pe = vivaldi_pattern(theta, Wa, L, Ri, 'E')
        ph = vivaldi_pattern(theta, Wa, L, Ri, 'H')
        pe_dB = 20 * np.log10(pe + 1e-10)
        ph_dB = 20 * np.log10(ph + 1e-10)
        hp_e = np.where(pe_dB >= -3)[0]
        hp_h = np.where(ph_dB >= -3)[0]
        bw_e = np.degrees(theta[hp_e[-1]] - theta[hp_e[0]]) if len(hp_e) >= 2 else 180
        bw_h = np.degrees(theta[hp_h[-1]] - theta[hp_h[0]]) if len(hp_h) >= 2 else 360
        hpbw_E.append(bw_e)
        hpbw_H.append(bw_h)
        D = 41253 / (abs(bw_e) * abs(bw_h) + 1e-6)
        gains.append(10 * np.log10(min(D, 100) + 1e-10))

    with out_taper:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.plot(Rs, hpbw_E, 'o-', color='tomato', markersize=3, label='E-plane')
        ax1.plot(Rs, hpbw_H, 's-', color='steelblue', markersize=3, label='H-plane')
        ax1.set_xlabel('Taper rate R'); ax1.set_ylabel('HPBW (°)')
        ax1.set_title(f'Beamwidth vs taper rate   L={L:.1f}λ  W_a={Wa:.1f}λ')
        ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

        ax2.plot(Rs, gains, 'o-', color='gold', markersize=3)
        ax2.set_xlabel('Taper rate R'); ax2.set_ylabel('Directivity (dBi)')
        ax2.set_title('Directivity vs taper rate')
        ax2.grid(alpha=0.3)

        plt.tight_layout(); plt.show()

for w in [w_trL, w_trWa]:
    w.observe(draw_taper_effect, names='value')
draw_taper_effect()
display(VBox([HBox([w_trL, w_trWa]), out_taper]))

## 9 — Vivaldi Array Element

Vivaldi antennas are commonly used as **array elements** in wideband phased arrays.  
Multiple Vivaldi elements side-by-side form an E-plane array:

$$AF(\theta) = \sum_{n=0}^{N-1} e^{\,jnkd_e\sin\theta}$$

where $d_e$ is the element spacing (typically $\approx \lambda/2$ at the centre frequency).

The total pattern = **element pattern** × **array factor**.

In [ ]:
w_aN  = IntSlider(value=4, min=1, max=8, step=1,
                  description='Elements', continuous_update=False,
                  style={'description_width': '100px'})
w_ade = FloatSlider(value=0.5, min=0.3, max=1.5, step=0.05,
                    description='Spacing d_e/λ', continuous_update=False,
                    style={'description_width': '100px'})
w_aR  = FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05,
                    description='Taper rate R', continuous_update=False,
                    style={'description_width': '100px'})
w_aL  = FloatSlider(value=4.0, min=1.0, max=8.0, step=0.1,
                    description='Length L/λ', continuous_update=False,
                    style={'description_width': '100px'})
w_ascan = FloatSlider(value=0.0, min=-60, max=60, step=1,
                      description='Scan θ₀ (°)', continuous_update=False,
                      style={'description_width': '100px'})
out_arr = Output()

def draw_array(change=None):
    N = w_aN.value; de = w_ade.value; R = w_aR.value
    L = w_aL.value; Wa = 1.5; scan = np.radians(w_ascan.value)

    theta = np.linspace(-np.pi, np.pi, 720)
    k = 2 * np.pi

    # element pattern (H-plane, since elements are arrayed along H)
    elem_pat = vivaldi_pattern(theta, Wa, L, R, 'H')

    # array factor
    delta = -k * de * np.sin(scan)
    psi = k * de * np.sin(theta) + delta
    with np.errstate(divide='ignore', invalid='ignore'):
        AF = np.where(np.abs(np.sin(psi / 2)) < 1e-12, 1.0,
                      np.abs(np.sin(N * psi / 2) / (N * np.sin(psi / 2))))

    total = elem_pat * AF
    total /= total.max() + 1e-30

    theta_deg = np.degrees(theta)
    total_dB = 20 * np.log10(total + 1e-10)
    elem_dB = 20 * np.log10(elem_pat + 1e-10)
    AF_dB = 20 * np.log10(AF + 1e-10)

    with out_arr:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

        # antenna layout sketch
        ys = (np.arange(N) - (N - 1) / 2) * de
        x_taper = np.linspace(0, L, 100)
        Wt = 0.05
        hw_taper = vivaldi_taper(x_taper, L, Wa, Wt, R)
        for yi in ys:
            ax1.fill_between(x_taper, yi - hw_taper * 0.3, yi + hw_taper * 0.3,
                             color='goldenrod', alpha=0.5)
            ax1.plot(x_taper, yi + hw_taper * 0.3, 'k-', lw=0.8)
            ax1.plot(x_taper, yi - hw_taper * 0.3, 'k-', lw=0.8)
        ax1.set_xlabel('x / λ'); ax1.set_ylabel('y / λ')
        ax1.set_title(f'{N}-element Vivaldi array   d_e={de:.2f}λ')
        ax1.set_aspect('equal')
        ax1.grid(alpha=0.2)

        # pattern
        ax2.plot(theta_deg, total_dB, lw=2, color='gold', label='Total')
        ax2.plot(theta_deg, elem_dB, lw=1, color='steelblue', alpha=0.5, label='Element')
        ax2.plot(theta_deg, AF_dB, lw=1, color='tomato', alpha=0.5, label='AF')
        ax2.axvline(w_ascan.value, color='gray', ls=':', lw=1)
        ax2.set_xlim(-180, 180); ax2.set_ylim(-40, 3)
        ax2.set_xlabel('θ (°)'); ax2.set_ylabel('Pattern (dB)')
        ax2.set_title(f'Array pattern   scan={w_ascan.value:.0f}°')
        ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

        plt.tight_layout(); plt.show()

for w in [w_aN, w_ade, w_aR, w_aL, w_ascan]:
    w.observe(draw_array, names='value')
draw_array()
display(VBox([HBox([w_aN, w_ade, w_ascan]),
              HBox([w_aR, w_aL]), out_arr]))